In [1]:
# ============================================
# 0. 載入套件與環境設定
# ============================================

import os
import sys

current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from train_save import train_and_save_model

print("環境準備完畢，已成功載入 train_and_save_model 模組。")

環境準備完畢，已成功載入 train_and_save_model 模組。


In [2]:
# ============================================
# 定義 Pydantic 訓練相關模型（與 app.py 相同）
# ============================================

from pydantic import BaseModel,Field
from pprint import pprint

class TrainConfig(BaseModel):
    test_size: float = Field(0.2, description="測試集分割比例", ge=0.1 , le=0.5)
    random_state: int = Field(76, description="隨機種子", ge=0)
    model_type: str = Field("LinearRegression", description="模型演算法類型 (LinearRegression, Lasso, Ridge)")
    alpha: float = Field(1.0, description="正則化強度 alpha (適用於 Lasso 與 Ridge)", ge= 0.001, le=100.0)

class TrainResult(BaseModel):
    status: str = Field(..., description="執行結果狀態")
    r2: float = Field(..., description="測試集 R-squared 決定係數")
    coef: list[float] = Field(..., description="特徵權重係數列表")
    intercept: float = Field(..., description="截距")
    feature_coefs: dict[str, float] = Field(..., description="特徵及其權重映射")
    model_type: str = Field(..., description="模型演算法類型")
    alpha: float = Field(..., description="正則化強度 alpha")
    train_time: float = Field(..., description="訓練耗時 (秒)")
    message:str = Field(..., description="提示訊息")

print("TranConfig(BaseModel)")
pprint(TrainConfig.model_json_schema())
print("==============================")
print("TrainResult(BaseModel)")
pprint(TrainResult.model_json_schema())

TranConfig(BaseModel)
{'properties': {'alpha': {'default': 1.0,
                          'description': '正則化強度 alpha (適用於 Lasso 與 Ridge)',
                          'maximum': 100.0,
                          'minimum': 0.001,
                          'title': 'Alpha',
                          'type': 'number'},
                'model_type': {'default': 'LinearRegression',
                               'description': '模型演算法類型 (LinearRegression, '
                                              'Lasso, Ridge)',
                               'title': 'Model Type',
                               'type': 'string'},
                'random_state': {'default': 76,
                                 'description': '隨機種子',
                                 'minimum': 0,
                                 'title': 'Random State',
                                 'type': 'integer'},
                'test_size': {'default': 0.2,
                              'description': '測試集分割比例',
              

In [3]:
class SalaryInput(BaseModel):
    years_experience: float = Field(..., ge=0.0, le=50.0)
    education_level:str
    city: str

class SalaryOutput(BaseModel):
    predicted_salary: float
    estimated_annual_salary: float
    

In [4]:
from train_save import train_and_save_model
res_ridge:dict = train_and_save_model(
    test_size=0.2,
    random_state=76,
    model_type="Ridge",
    alpha=10.0
)
pprint(res_ridge)

開始訓練 Ridge 嶺迴歸(α=10.0) (測試集比例:0.2, 隨機種子:76)....
正在將模型、預處理器與元數據序列化並儲存至 C:\Users\User\Documents\GitHub\2026-07-03\banend\08-07\salary_model.joblib...
模型儲存成功！
{'alpha': 10.0,
 'coef': [3.915606705818322,
          10.029103401270465,
          -1.4644383465780102,
          -1.182860911975303,
          2.1482340576072554],
 'feature_coefs': {'City_城市A': -1.4644383465780102,
                   'City_城市B': -1.182860911975303,
                   'City_城市C': 2.1482340576072554,
                   'EducationLevel': 10.029103401270465,
                   'YearsExperience': 3.915606705818322},
 'intercept': 51.228571428571435,
 'message': 'Ridge 嶺迴歸(α=10.0) 模型訓練完成並儲存成功！',
 'model_type': 'Ridge',
 'r2': 0.8253872705107945,
 'status': 'success',
 'train_time': 0.0590972900390625}


In [5]:
import joblib

current_dir = os.getcwd()
model_path = os.path.join(current_dir, "salary_model.joblib")
MODEL_STATE = {}

def load_model_state():
    global MODEL_STATE
    if not os.path.exists(model_path):
        train_and_save_model()

    model_data = joblib.load(model_path)
    MODEL_STATE.clear()
    MODEL_STATE.update(
        {
            "model": model_data["model"],
            "oe": model_data["oe"],
            "ohe": model_data["ohe"],
            "scaler": model_data["scaler"],
            "r2": model_data.get("r2"),
            "feature_names": model_data["feature_names"],
            "feature_coefs": model_data.get("feature_coefs",{}),
            "model_type": model_data.get("model_type"),
            "alpha": model_data.get("alpha")
        }
    )
    print(f"✅ MODEL_STATE 已成功更新！當前模型：{MODEL_STATE['model_type']}，R² Score：{MODEL_STATE['r2']:.4f}")

load_model_state()

✅ MODEL_STATE 已成功更新！當前模型：Ridge，R² Score：0.8254


In [6]:
from fastapi import HTTPException

def train_api(config:TrainConfig) -> dict:
    """
    訓練端點：傳入測試集比例、隨機種子、模型類型與 alpha，線上重新訓練模型，並即時更新服務所使用的模型。
    """
    try:
        # 1. 執行重新訓練並儲存模型
        res = train_and_save_model(
            test_size=config.test_size,
            random_state= config.random_state,
            model_type= config.model_type,
            alpha=config.alpha
        )
         # 2. 線上重新載入最新模型狀態至全域變數
        load_model_state()
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"線上訓練失敗: {str(e)}")

    return res

In [7]:
import pandas as pd
import numpy as np

def predict_api(years_experience:float, education_level:str, city:str) -> dict:
    oe = MODEL_STATE["oe"]
    ohe = MODEL_STATE["ohe"]
    scaler = MODEL_STATE["scaler"]
    model = MODEL_STATE["model"]

    edu_encoded = int(oe.transform(pd.DataFrame([[education_level]], columns=["EducationLevel"]))[0][0])
    city_vector = ohe.transform(pd.DataFrame([[city]], columns=["City"]))
    city_cols = ohe.get_feature_names_out(['City'])
    feature_row = [years_experience, edu_encoded] + list(city_vector[0])
    features = pd.DataFrame([feature_row],columns=["YearsExperience", "EducationLevel"] + list(city_cols))
    X_scaled = scaler.transform(features)
    predicted_salary = float(model.predict(X_scaled)[0])
    return {
        "predicted_salary": predicted_salary,
        "estimated_annual_salary": predicted_salary * 14
    }


predict_api(years_experience=10.0, education_level="大學", city="城市C")

{'predicted_salary': 61.07802879415863,
 'estimated_annual_salary': 855.0924031182208}

In [8]:
from fastapi.testclient import TestClient
from pprint import pprint

# 💡 只保留最純粹的測試邏輯，直接從你前面已經執行成功的環境抓取模型
client = TestClient(mini_api)

# 1. 測試模型重訓
response = client.post("/train", json={
    "test_size": 0.2,
    "random_state": 76,
    "model_type": "Lasso",
    "alpha": 5.0
})
print("【重訓 Lasso 結果】")
print("HTTP 狀態碼:", response.status_code)

# 💡 直接印出 response 的內容，不經過任何 response_model 過濾，保證 feature_coefs 一定在裡面！
pprint(response.json())

# 2. 測試薪資預測
response1 = client.post("/predict", json={
    "years_experience": 5.3,
    "education_level": "碩士以上",
    "city": "城市A"
})

print("\n預測月薪:", response1.json()["predicted_salary"])


NameError: name 'mini_api' is not defined